# RAG ToT  
Tree-of-Thought 기법을 적용

- RAG에 Tree-of-Thought(ToT) 기법을 적용하면,  
  단일 답변 생성이 아닌 **여러 추론 경로(후보 답변)를 생성·평가·선택**하는 구조로 확장할 수 있다.
- 각 Thought(노드)는 **조회된 문서 기반의 부분 추론 결과**를 의미하며,  
  이를 단계적으로 분기·탐색함으로써 답변의 정확성과 근거성을 높인다.
- 이를 통해 RAG는 단순 Q&A를 넘어  
  **복합 질문, 비교·분석형 질문에 대응 가능한 추론형 검색 시스템**으로 발전한다.

In [1]:
%pip install langchain langchain-openai -Uq

Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")

In [6]:
from langchain_core.documents import Document  # LangChain Document 클래스 import

# 더미 벡터DB에서 경제 관련 Document 리스트를 반환하는 함수
def retrieve_vectordb(query=None):
    return [  # Document 객체 리스트 반환
        Document(  # 1번 문서 생성
            page_content="""  
정부는 2026년을 '한국경제 대도약 원년'으로 선포하고, 0%대까지 추락했던 잠재성장률을 다시 1.8% 이상으로 끌어올리기 위한 적극적 확장 재정 정책을 추진하고 있습니다.
기존의 감세 중심에서 '지출 중심'으로 전환하며, 내년도 성장률 목표 달성을 위한 대규모 예산 투입을 공식화했습니다.
규제 측면에서는 금산분리 완화 등 실용주의적 개혁을 추진하며, 한국은행은 2026년 1월 기준금리를 연 2.50%로 동결하며 신중한 기조를 유지 중입니다.
""",  # 문서 본문(정책 방향)
            metadata={  # 문서 메타데이터
                "source": "DOC1",  # 문서 ID/출처
                "title": "2026년 경제정책 방향",  # 문서 제목
                "category": "Policy"  # 문서 분류
            }
        ),
        Document(  # 2번 문서 생성
            page_content="""  
2026년 한국 경제는 수출 증가세가 둔화되는 가운데 내수가 완만한 회복세를 보이며 1.8% 수준의 성장을 기록할 것으로 전망됩니다.
이는 2025년의 저성장(0.7~0.8%)에서 벗어나 정상 궤도로 진입하는 과정입니다.
반도체, AI, 조선, 방산 분야는 긍정적이나 미·중 통상 갈등 및 지정학적 리스크로 인한 수출 불확실성은 여전히 높은 상황입니다.
""",  # 문서 본문(전망)
            metadata={  # 문서 메타데이터
                "source": "DOC2",  # 문서 ID/출처
                "title": "2026년 경제 전망",  # 문서 제목
                "category": "Forecast"  # 문서 분류
            }
        ),
        Document(  # 3번 문서 생성
            page_content="""  
2025년의 저성장 충격을 극복하기 위해 정부는 양극화 구조 타파와 지속 가능한 성장에 집중하고 있습니다.
대·중소기업 상생, 지역 균형 발전, 노동시장 이중구조 완화를 핵심 과제로 설정하였습니다.
또한 탄소중립, 에너지 전환 등 ESG 가치를 반영한 R&D 혁신과 AI 대전환을 통해 장기적인 국가 경쟁력 확보에 주력하고 있습니다.
""",  # 문서 본문(전략)
            metadata={  # 문서 메타데이터
                "source": "DOC3",  # 문서 ID/출처
                "title": "경기 침체 탈출 및 양극화 해소",  # 문서 제목
                "category": "Strategy"  # 문서 분류
            }
        )
    ]

# 데이터 확인
docs = retrieve_vectordb()  # 문서 리스트 로드(더미 벡터DB 조회)
docs

[Document(metadata={'source': 'DOC1', 'title': '2026년 경제정책 방향', 'category': 'Policy'}, page_content="  \n정부는 2026년을 '한국경제 대도약 원년'으로 선포하고, 0%대까지 추락했던 잠재성장률을 다시 1.8% 이상으로 끌어올리기 위한 적극적 확장 재정 정책을 추진하고 있습니다.\n기존의 감세 중심에서 '지출 중심'으로 전환하며, 내년도 성장률 목표 달성을 위한 대규모 예산 투입을 공식화했습니다.\n규제 측면에서는 금산분리 완화 등 실용주의적 개혁을 추진하며, 한국은행은 2026년 1월 기준금리를 연 2.50%로 동결하며 신중한 기조를 유지 중입니다.\n"),
 Document(metadata={'source': 'DOC2', 'title': '2026년 경제 전망', 'category': 'Forecast'}, page_content='  \n2026년 한국 경제는 수출 증가세가 둔화되는 가운데 내수가 완만한 회복세를 보이며 1.8% 수준의 성장을 기록할 것으로 전망됩니다.\n이는 2025년의 저성장(0.7~0.8%)에서 벗어나 정상 궤도로 진입하는 과정입니다.\n반도체, AI, 조선, 방산 분야는 긍정적이나 미·중 통상 갈등 및 지정학적 리스크로 인한 수출 불확실성은 여전히 높은 상황입니다.\n'),
 Document(metadata={'source': 'DOC3', 'title': '경기 침체 탈출 및 양극화 해소', 'category': 'Strategy'}, page_content='  \n2025년의 저성장 충격을 극복하기 위해 정부는 양극화 구조 타파와 지속 가능한 성장에 집중하고 있습니다.\n대·중소기업 상생, 지역 균형 발전, 노동시장 이중구조 완화를 핵심 과제로 설정하였습니다.\n또한 탄소중립, 에너지 전환 등 ESG 가치를 반영한 R&D 혁신과 AI 대전환을 통해 장기적인 국가 경쟁력 확보에 주력하고 있습니다.\n')]

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from typing import List, Literal


class ExpertOpinion(BaseModel):
    role: str = Field(description='전문가 역할 (예: 국제거시경제전문가)')
    analysis : str = Field(description='해당 전문가의 상세 분석 내용')
    verdict : Literal['긍정', '부정', '중립'] = Field(description='분석 결과에 대한 종합 평가')
    
class FinalOpinion(BaseModel):
    opinions:List[ExpertOpinion] = Field(description='개별 전문가들의 의견 리스트')
    final_analysis : str = Field(description='전문가의 의견을 종합한 최종 분석내용')
    final_verdict: Literal['긍정', '부정','중립'] = Field(description='전문가의 의견을 종합한 최종 평가')

llm = init_chat_model('openai:gpt-4.1-mini')

prompt = prompt = PromptTemplate.from_template('''  # 다중 관점 분석 및 결론 도출 프롬프트
당신은 현실세계의 복잡한 경제문제를 해결하기 위해 다양한 관점으로 바라보고, 분석하는 에이젼트입니다.
다음 세가지 관점에서 주어진 문서를 분석하고, 이를 종합하여 결론을 작성해주세요.

1. 분석가1 (국제거시경제): 글로벌 트렌드와 거시지표(성장률, 금리) 관점에서 분석 전문가 (긍정/중립/부정)
2. 분석가2 (지역경제/내수): 내수 소비, 건설, 지역경제활성화 관점에서 분석 전문가 (긍정/중립/부정)
3. 분석가3 (ESG/지속가능성): 장기적 안정성과 환경/사회적 영향 관점에서 분석 전문가 (긍정/중립/부정)

위 분석가들의 의견을 토대로 최종결론을 도출해주세요.  (긍정/중립/부정)

[문서]
{context}

[사용자 질문]
{query}
''')

chain = prompt | llm.with_structured_output(FinalOpinion)   # 응답을 FinalOpinion 스키마로 강제

query = '현재 한국 정부의 경제 정책을 어떻게 봐야 할까?'

context = '\n\n'.join([doc.page_content for doc in retrieve_vectordb(query)])   # 조회 문서 본문 1개의 문자열로 결합


final_opinion = chain.invoke({'query' : query , 'context' : context})

print(final_opinion)



opinions=[ExpertOpinion(role='분석가1 (국제거시경제)', analysis="한국 정부는 2026년을 '한국경제 대도약 원년'으로 선포하고 0%대 잠재성장률을 1.8% 이상으로 회복하기 위해 적극적인 확장 재정 정책을 추진 중입니다. 금리는 2.5%로 동결하여 신중한 기조를 유지함에도 불구하고 글로벌 통상 리스크(미·중 갈등 등)가 존재하지만 주요 산업(반도체, AI, 조선, 방산)의 성장 가능성은 긍정적입니다. 전반적으로 글로벌 거시경제 흐름과 위험요인을 감안할 때, 정책은 경제성장 회복에 대한 긍정적인 기여가 기대됩니다.", verdict='긍정'), ExpertOpinion(role='분석가2 (지역경제/내수)', analysis='내수 소비가 완만한 회복세를 보이고 있으며, 정부가 내수 및 지역경제 활성화를 위해 지출 중심의 대규모 예산을 투입하는 점은 긍정적입니다. 대·중소기업 상생과 지역 균형 발전, 노동시장 이중구조 완화 등을 핵심과제로 삼는 것은 내수 확대 및 지역 경제 활성화에 도움될 것으로 평가됩니다. 다만, 수출 둔화와 저성장 충격 후유증은 단기적으로 내수 회복을 제한할 수 있으나 정책 방향은 내수 진작에 적절합니다.', verdict='긍정'), ExpertOpinion(role='분석가3 (ESG/지속가능성)', analysis='정부가 양극화 해소, 대·중소기업 상생, 탄소중립과 에너지 전환, ESG 가치를 반영한 R&D 혁신 및 AI 도입에 집중하는 점은 장기적인 지속가능성과 안정성 측면에서 매우 긍정적입니다. ESG 정책과 사회적 책임 강화가 국가 경쟁력 확보와 환경 보호에 기여할 것으로 보입니다. 다만 이러한 변화가 단기적 경제 충격을 완화하는 데는 한계가 있으나 장기적으로는 긍정적인 효과가 예상됩니다.', verdict='긍정')] final_analysis='세 가지 다양한 관점 모두 정부의 2026년 경제정책에 대해 긍정적인 평가를 내리고 있습니다. 국제 거시경제 측면에서 성장 회복 가능성을, 내수 및

In [ ]:
# 개별 분석가의 의견
for opinion in final_opinion.opinions:
    print(f"[{opinion.role} 의견 : {opinion.verdict}]") # 전문가 역할/평가 라벨
    print(opinion.analysis)                             # 전문가 상세 분석
    print()
    
print('-' * 100)
print(f"[최종 의견 :{final_opinion.final_verdict}]")    # 종합 평가 라벨 출력
print(f"{final_opinion.final_analysis}")                # 최종 종합 분석

[분석가1 (국제거시경제) 의견 : 긍정]
한국 정부는 2026년을 '한국경제 대도약 원년'으로 선포하고 0%대 잠재성장률을 1.8% 이상으로 회복하기 위해 적극적인 확장 재정 정책을 추진 중입니다. 금리는 2.5%로 동결하여 신중한 기조를 유지함에도 불구하고 글로벌 통상 리스크(미·중 갈등 등)가 존재하지만 주요 산업(반도체, AI, 조선, 방산)의 성장 가능성은 긍정적입니다. 전반적으로 글로벌 거시경제 흐름과 위험요인을 감안할 때, 정책은 경제성장 회복에 대한 긍정적인 기여가 기대됩니다.

[분석가2 (지역경제/내수) 의견 : 긍정]
내수 소비가 완만한 회복세를 보이고 있으며, 정부가 내수 및 지역경제 활성화를 위해 지출 중심의 대규모 예산을 투입하는 점은 긍정적입니다. 대·중소기업 상생과 지역 균형 발전, 노동시장 이중구조 완화 등을 핵심과제로 삼는 것은 내수 확대 및 지역 경제 활성화에 도움될 것으로 평가됩니다. 다만, 수출 둔화와 저성장 충격 후유증은 단기적으로 내수 회복을 제한할 수 있으나 정책 방향은 내수 진작에 적절합니다.

[분석가3 (ESG/지속가능성) 의견 : 긍정]
정부가 양극화 해소, 대·중소기업 상생, 탄소중립과 에너지 전환, ESG 가치를 반영한 R&D 혁신 및 AI 도입에 집중하는 점은 장기적인 지속가능성과 안정성 측면에서 매우 긍정적입니다. ESG 정책과 사회적 책임 강화가 국가 경쟁력 확보와 환경 보호에 기여할 것으로 보입니다. 다만 이러한 변화가 단기적 경제 충격을 완화하는 데는 한계가 있으나 장기적으로는 긍정적인 효과가 예상됩니다.

----------------------------------------------------------------------------------------------------
[최종 의견 :긍정]
세 가지 다양한 관점 모두 정부의 2026년 경제정책에 대해 긍정적인 평가를 내리고 있습니다. 국제 거시경제 측면에서 성장 회복 가능성을, 내수 및 지역경제 활성화 측면에서는 정책의 실효성을, E